In [1]:
import scanpy as sc
import scarches
from scarches.models.scpoli import scPoli
import scib
import os
import anndata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import issparse
import scipy.sparse as sparse
import warnings
os.getcwd()

# Filter out DeprecationWarnings
#warnings.filterwarnings("ignore", category=DeprecationWarning)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scvi/_settings.py:63: UserWarning: Since v1.0.0, scvi-tools no longer uses a random seed by default. Run `scvi.settings.seed = 0` to reproduce results from previous versions.
  self.seed = seed
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scvi/_settings.py:70: UserWarning: Setting `dl_pin_memory_gpu_training` is deprecated in v1.0 and will be removed in v1.1. Please pass in `pin_memory` to the data loaders instead.
  self.dl_pin_memory_gpu_training = (
 captum (see https://github.com/pytorch/captum).


'/home/lixiangyu/zr/Annotate/ANNOTATE_new/7.5_correct/0511_rename_noIAISR/train_and_R'

## preprocessing: HVG, PCA, checks

In [ ]:
adata_final = sc.read_h5ad("/home/lixiangyu/zr/Annotate/ANNOTATE_new/7.5_correct/0511_rename_noIAISR/output_allhuman/work_0513_2/scPoli_concat_corrected_all.h5ad")
adata_final

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 1018141 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'data_role', 'species'
    var: 'original_gene_names', 'ensembl_id'
    layers: 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'

In [3]:
# def preprocess_adata(adata_final, mode, batch_key):

#     adata_final = adata_final.copy()

#     adata_final.obs[batch_key] = adata_final.obs[batch_key].astype("category")
#     adata_final.obs[batch_key] = adata_final.obs[batch_key].cat.remove_unused_categories()

#     if "counts" not in adata_final.layers:
#         if "rounded_corrected_counts" in adata_final.layers:
#             adata_final.layers["counts"] = adata_final.layers["rounded_corrected_counts"].copy()
#         else:
#             raise ValueError("No counts layer found.")

#     # 关键：从 counts 重新构建干净的 log-normalized .X
#     print("Rebuilding adata.X from counts...")
#     adata_final.X = adata_final.layers["counts"].copy()
#     sc.pp.normalize_total(adata_final, target_sum=1e4)
#     sc.pp.log1p(adata_final)

#     if mode == "auto":
#         print("Using auto mode to do hvg and pca...")

#         hvg_list = scib.preprocessing.hvg_batch(
#             adata_final,
#             batch_key=batch_key,
#             target_genes=2000,
#             flavor="seurat"
#         )

#         adata_final.var["highly_variable"] = adata_final.var_names.isin(hvg_list)

#         sc.pp.pca(
#             adata_final,
#             use_highly_variable=True,
#             n_comps=50
#         )

#     elif mode == "manual":
#         print("Using manual mode to do hvg and pca...")

#         sc.pp.highly_variable_genes(
#             adata_final,
#             n_top_genes=2000,
#             batch_key=batch_key,
#             flavor="seurat"
#         )

#         sc.pp.pca(
#             adata_final,
#             use_highly_variable=True,
#             n_comps=50
#         )

#     else:
#         raise ValueError("Mode must be 'auto' or 'manual'")

#     print("Subset for HVGs...")
#     hvg = adata_final.var[adata_final.var["highly_variable"]].index.tolist()
#     adata_final = adata_final[:, hvg].copy()

#     print("Check how many cells have zero counts for all genes...")
#     cellwise_sum = np.asarray(adata_final.X.sum(axis=1)).ravel()
#     num_cells_zero_counts = int((cellwise_sum == 0).sum())

#     if num_cells_zero_counts > 0:
#         print(num_cells_zero_counts, "cells were found with 0 expression across all HVGs! Removing these cells now...")
#         adata_final = adata_final[cellwise_sum > 0, :].copy()

#     print("Check for duplicate gene expressions")

#     counts = adata_final.layers["counts"]
#     counts_array = counts.toarray() if sparse.issparse(counts) else np.asarray(counts)

#     _, unique_indices = np.unique(counts_array, axis=0, return_index=True)
#     unique_indices_sorted = np.sort(unique_indices)

#     diff = counts_array.shape[0] - len(unique_indices_sorted)

#     if diff > 0:
#         print(diff, "non-unique cell expression profiles found! Removing them...")
#         adata_final = adata_final[unique_indices_sorted, :].copy()
#     else:
#         print("No non-unique cells found.")

#     print("Adding atlas key")
#     adata_final.obs["atlas_key"] = [
#         "query" if cell_type == "unknown" else "ref"
#         for cell_type in adata_final.obs["cell_type_level1"]
#     ]

#     print("Preprocessing finished!")

#     return adata_final

In [ ]:
# #不转密集
# def preprocess_adata(adata_final, mode, batch_key):

#     def _unique_row_indices(mat):
#         if sparse.issparse(mat):
#             mat = mat.tocsr(copy=True)
#             mat.sum_duplicates()
#             mat.sort_indices()
#             seen = set()
#             unique_indices = []
#             for i in range(mat.shape[0]):
#                 start, end = mat.indptr[i], mat.indptr[i + 1]
#                 key = (tuple(mat.indices[start:end]), tuple(mat.data[start:end]))
#                 if key not in seen:
#                     seen.add(key)
#                     unique_indices.append(i)
#             return np.array(unique_indices, dtype=int)
#         else:
#             mat = np.asarray(mat)
#             _, unique_indices = np.unique(mat, axis=0, return_index=True)
#             return np.sort(unique_indices)

#     adata_final = adata_final.copy()

#     adata_final.obs[batch_key] = adata_final.obs[batch_key].astype("category")
#     adata_final.obs[batch_key] = adata_final.obs[batch_key].cat.remove_unused_categories()

#     if "counts" not in adata_final.layers:
#         if "rounded_corrected_counts" in adata_final.layers:
#             adata_final.layers["counts"] = adata_final.layers["rounded_corrected_counts"].copy()
#         else:
#             raise ValueError("No counts layer found.")

#     # 关键：从 counts 重新构建干净的 log-normalized .X
#     print("Rebuilding adata.X from counts...")
#     adata_final.X = adata_final.layers["counts"].copy()
#     sc.pp.normalize_total(adata_final, target_sum=1e4)
#     sc.pp.log1p(adata_final)

#     if mode == "auto":
#         print("Using auto mode to do hvg and pca...")

#         hvg_list = scib.preprocessing.hvg_batch(
#             adata_final,
#             batch_key=batch_key,
#             target_genes=2000,
#             flavor="seurat"
#         )

#         adata_final.var["highly_variable"] = adata_final.var_names.isin(hvg_list)

#         sc.pp.pca(
#             adata_final,
#             use_highly_variable=True,
#             n_comps=50
#         )

#     elif mode == "manual":
#         print("Using manual mode to do hvg and pca...")

#         sc.pp.highly_variable_genes(
#             adata_final,
#             n_top_genes=2000,
#             batch_key=batch_key,
#             flavor="seurat"
#         )

#         sc.pp.pca(
#             adata_final,
#             use_highly_variable=True,
#             n_comps=50
#         )

#     else:
#         raise ValueError("Mode must be 'auto' or 'manual'")

#     print("Subset for HVGs...")
#     hvg = adata_final.var[adata_final.var["highly_variable"]].index.tolist()
#     adata_final = adata_final[:, hvg].copy()

#     print("Check how many cells have zero counts for all genes...")
#     cellwise_sum = np.asarray(adata_final.X.sum(axis=1)).ravel()
#     num_cells_zero_counts = int((cellwise_sum == 0).sum())

#     if num_cells_zero_counts > 0:
#         print(num_cells_zero_counts, "cells were found with 0 expression across all HVGs! Removing these cells now...")
#         adata_final = adata_final[cellwise_sum > 0, :].copy()

#     print("Check for duplicate gene expressions")

#     counts = adata_final.layers["counts"]
#     unique_indices_sorted = _unique_row_indices(counts)
#     diff = counts.shape[0] - len(unique_indices_sorted)

#     if diff > 0:
#         print(diff, "non-unique cell expression profiles found! Removing them...")
#         adata_final = adata_final[unique_indices_sorted, :].copy()
#     else:
#         print("No non-unique cells found.")
# #
#     print("Adding atlas key")
#     adata_final.obs["atlas_key"] = [
#         "query" if cell_type == "unknown" else "ref"
#         for cell_type in adata_final.obs["cell_type_level1"]
#     ]

#     print("Preprocessing finished!")

#     return adata_final

In [5]:
def preprocess_adata(adata_final, mode, batch_key, copy=False):
    import gc
    import hashlib
    import numpy as np
    from scipy import sparse

    def _unique_row_indices(mat):
        if sparse.issparse(mat):
            mat = mat.tocsr(copy=False)
            mat.sum_duplicates()
            mat.sort_indices()
            seen = set()
            unique_indices = []
            for i in range(mat.shape[0]):
                start, end = mat.indptr[i], mat.indptr[i + 1]
                h = hashlib.blake2b(digest_size=16)
                h.update(mat.indices[start:end].tobytes())
                h.update(mat.data[start:end].tobytes())
                key = h.digest()
                if key not in seen:
                    seen.add(key)
                    unique_indices.append(i)
            return np.asarray(unique_indices, dtype=np.int64)
        else:
            mat = np.asarray(mat)
            _, unique_indices = np.unique(mat, axis=0, return_index=True)
            return np.sort(unique_indices)

    if copy:
        adata_final = adata_final.copy()

    adata_final.obs[batch_key] = adata_final.obs[batch_key].astype("category").cat.remove_unused_categories()

    if "counts" not in adata_final.layers:
        if "rounded_corrected_counts" in adata_final.layers:
            adata_final.layers["counts"] = adata_final.layers["rounded_corrected_counts"]
        else:
            raise ValueError("No counts layer found.")

    print("Rebuilding adata.X from counts...")
    X = adata_final.layers["counts"]
    if sparse.issparse(X):
        adata_final.X = X.copy().astype(np.float32)
    else:
        adata_final.X = np.asarray(X, dtype=np.float32).copy()

    sc.pp.normalize_total(adata_final, target_sum=1e4)
    sc.pp.log1p(adata_final)

    if mode == "auto":
        print("Using auto mode to do hvg...")
        hvg_list = scib.preprocessing.hvg_batch(adata_final, batch_key=batch_key, target_genes=2000, flavor="seurat")
        adata_final.var["highly_variable"] = adata_final.var_names.isin(hvg_list)
    elif mode == "manual":
        print("Using manual mode to do hvg...")
        sc.pp.highly_variable_genes(adata_final, n_top_genes=2000, batch_key=batch_key, flavor="seurat")
    else:
        raise ValueError("Mode must be 'auto' or 'manual'")

    print("Subset for HVGs before PCA...")
    hvg = adata_final.var_names[adata_final.var["highly_variable"]].tolist()
    adata_final = adata_final[:, hvg].copy()
    gc.collect()

    print("Running PCA...")
    sc.pp.pca(adata_final, n_comps=50)

    print("Check how many cells have zero counts for all genes...")
    cellwise_sum = np.asarray(adata_final.X.sum(axis=1)).ravel()
    keep = cellwise_sum > 0
    num_cells_zero_counts = int((~keep).sum())

    if num_cells_zero_counts > 0:
        print(num_cells_zero_counts, "cells were found with 0 expression across all HVGs! Removing these cells now...")
        adata_final = adata_final[keep, :].copy()
        gc.collect()

    print("Check for duplicate gene expressions")
    counts = adata_final.layers["counts"]
    unique_indices_sorted = _unique_row_indices(counts)
    diff = counts.shape[0] - len(unique_indices_sorted)

    if diff > 0:
        print(diff, "non-unique cell expression profiles found! Removing them...")
        adata_final = adata_final[unique_indices_sorted, :].copy()
        gc.collect()
    else:
        print("No non-unique cells found.")

    print("Adding atlas key")
    adata_final.obs["atlas_key"] = np.where(
        adata_final.obs["cell_type_level1"].astype(str).values == "unknown",
        "query",
        "ref"
    )

    print("Preprocessing finished!")
    return adata_final

In [6]:
adata_pp = preprocess_adata(adata_final, mode = "auto", batch_key="sample")

Rebuilding adata.X from counts...


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/preprocessing/_normalization.py:196: UserWarning: Some cells have zero counts
  warn(UserWarning('Some cells have zero counts'))


Using auto mode to do hvg...


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/preprocessing/_highly_variable_genes.py:220: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby('mean_bin')['dispersions']
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/preprocessing/_highly_variable_genes.py:220: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby('mean_bin')['dispersions']
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/preprocessing/_highly_variable_genes.py:220: FutureWarning: The default of observed=False is deprecated an

Using 0 HVGs from full intersect set
Using 0 HVGs from n_batch-1 set
Using 0 HVGs from n_batch-2 set
Using 0 HVGs from n_batch-3 set
Using 0 HVGs from n_batch-4 set
Using 0 HVGs from n_batch-5 set
Using 0 HVGs from n_batch-6 set
Using 0 HVGs from n_batch-7 set
Using 0 HVGs from n_batch-8 set
Using 0 HVGs from n_batch-9 set
Using 0 HVGs from n_batch-10 set
Using 1 HVGs from n_batch-11 set
Using 0 HVGs from n_batch-12 set
Using 1 HVGs from n_batch-13 set
Using 0 HVGs from n_batch-14 set
Using 1 HVGs from n_batch-15 set
Using 2 HVGs from n_batch-16 set
Using 3 HVGs from n_batch-17 set
Using 0 HVGs from n_batch-18 set
Using 3 HVGs from n_batch-19 set
Using 1 HVGs from n_batch-20 set
Using 1 HVGs from n_batch-21 set
Using 3 HVGs from n_batch-22 set
Using 3 HVGs from n_batch-23 set
Using 0 HVGs from n_batch-24 set
Using 1 HVGs from n_batch-25 set
Using 2 HVGs from n_batch-26 set
Using 4 HVGs from n_batch-27 set
Using 1 HVGs from n_batch-28 set
Using 3 HVGs from n_batch-29 set
Using 3 HVGs fr

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Running PCA...
Check how many cells have zero counts for all genes...
210 cells were found with 0 expression across all HVGs! Removing these cells now...
Check for duplicate gene expressions
2613 non-unique cell expression profiles found! Removing them...
Adding atlas key
Preprocessing finished!


In [ ]:
adata_pp.write_h5ad("/home/lixiangyu/zr/Annotate/ANNOTATE_new/7.5_correct/0511_rename_noIAISR/output_allhuman/work_0513_2/scPoli_concat_corrected_all_pp.h5ad")###合并的时候改的dc名字
adata_pp

AnnData object with n_obs × n_vars = 1015318 × 2000
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'data_role', 'species', 'atlas_key'
    var: 'original_gene_names', 'ensembl_id', 'highly_variable'
    uns: 'log1p', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'
    layers: 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts', 'counts'

In [8]:
adata_pp.obs['cell_type_level1'].value_counts()

cell_type_level1
T cell                   367267
Macrophage               112016
Fibroblast                91779
B cell                    90167
Neutrophil                78293
Smooth muscle cell        71639
Endothelial cell          64895
Monocyte                  42921
Natural killer cell       41251
Dendritic cell            21699
Pericyte                  18989
Mast cell                  6951
Basophil                   4106
Erythrocyte/Erythroid      3345
Name: count, dtype: int64